# Stages 3-4 report: training curves, val / test results, comparisons

Run this after `scripts/model/train_mil.py` (and `evaluate_mil.py` for test). It only reads the experiment
records in `outputs/experiments/`; nothing is trained or written.

1. **The ledger**: every finished run, one row each.
2. **One run**: learning curves (is the epoch budget enough?) and per-label val results.
3. **Its test results**: overall, sharp vs soft kernel (cross-kernel gap), per scanner.
4. **Comparisons**: configurations averaged over seeds (loss, single vs per-label attention, mean-pool baseline).

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# run from the repository root, wherever the notebook was opened
root = Path.cwd().resolve()
while not (root / "configs" / "preprocessing.yaml").exists() and root != root.parent:
    root = root.parent
os.chdir(root)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

## 1. The ledger

`RUN_DIR = None` picks the most recently finished run; otherwise give its folder
(`outputs/experiments/<name>/<config hash>/seed<k>`).

In [ ]:
OUTPUT_DIR = Path("outputs/experiments")
RUN_DIR = None

ledger = pd.read_csv(OUTPUT_DIR / "index.csv")
display(ledger[["run_id", "aggregator", "loss", "lr", "best_epoch", "epochs_run", "stopped",
                "val_macro_auroc", "val_macro_auprc", "val_macro_f1"]].sort_values("val_macro_auroc", ascending=False))
run = Path(RUN_DIR) if RUN_DIR else Path(ledger.sort_values("finished").dir.iloc[-1])
info = pd.read_json(run / "run_info.json", typ="series")
print("run:", info["run_id"], "| best epoch", info["best_epoch"], "of", info["epochs_run"], "run,", "stopped by", info["stopped"])

## 2. Learning curves and per-label val results

The best epoch is marked. If it sits in the last 10 % of `max_epochs`, the budget may be too small
(the curve was still improving): raise `optim.max_epochs` for that config.

In [ ]:
import yaml

m = pd.read_csv(run / "metrics.csv")
cfg = yaml.safe_load((run / "config.yaml").read_text())
best = int(info["best_epoch"])
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].plot(m.epoch, m.train_loss, label="train"); ax[0].plot(m.epoch, m.val_loss, label="val")
ax[0].set_title("loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot(m.epoch, m.val_macro_auroc, label="val macro AUROC"); ax[1].plot(m.epoch, m.val_macro_auprc, label="val macro AUPRC")
ax[1].set_title("val metrics"); ax[1].set_xlabel("epoch"); ax[1].legend()
for a in ax:
    a.axvline(best, color="gray", ls="--", lw=1)
plt.tight_layout(); plt.show()
budget = cfg["optim"]["max_epochs"]
print(f"best epoch {best} / budget {budget} epochs ({info['steps_per_epoch']} steps per epoch, best at step {best * info['steps_per_epoch']})")
if best > 0.9 * budget:
    print("-> the best epoch is in the last 10% of the budget: consider raising optim.max_epochs")

In [ ]:
val = pd.read_csv(run / "val_per_label.csv").sort_values("auroc")
display(val.round(3))
fig, ax = plt.subplots(figsize=(8, 0.35 * len(val) + 1))
ax.barh(val.label, val.auroc, color="#4C78A8"); ax.set_xlim(0.5, 1); ax.set_xlabel("val AUROC (best epoch)")
for y, (v, n) in enumerate(zip(val.auroc, val.n_pos)):
    ax.text(v + 0.003, y, f"{v:.3f} (n+={n})", va="center", fontsize=8)
plt.tight_layout(); plt.show()

## 3. Test results of this run

Only after `evaluate_mil.py`. Thresholds were chosen on val. Training saw only sharp kernels, so
**sharp vs soft** is the cross-kernel result; groups flagged `too_small` should not be read alone.

In [ ]:
evals = sorted((run / "eval").glob("*__test")) if (run / "eval").is_dir() else []
if not evals:
    print("no test evaluation yet -- python scripts/model/evaluate_mil.py --experiment-dir", run.as_posix())
else:
    ev = evals[0]
    summary = pd.read_csv(ev / "metrics_summary.csv")
    cols = [c for c in ["group_by", "group", "n_volumes", "n_patients", "macro_auroc", "macro_auroc_ci_low",
                        "macro_auroc_ci_high", "macro_auprc", "macro_f1", "macro_sensitivity", "macro_specificity", "too_small"]
            if c in summary.columns]
    display(summary[cols].round(4))
    per = pd.read_csv(ev / "metrics_per_label.csv")
    k = per[per.group_by == "kernel_class"].pivot(index="label", columns="group", values="auroc")
    k = k[[c for c in ("sharp", "soft") if c in k.columns]].sort_values(k.columns[0])
    if len(k.columns) == 2:
        k["soft - sharp"] = k["soft"] - k["sharp"]
    display(k.round(3))
    ax = k[[c for c in ("sharp", "soft") if c in k.columns]].plot.barh(figsize=(8, 0.35 * len(k) + 1), width=0.8)
    ax.set_xlim(0.5, 1); ax.set_xlabel("test AUROC"); ax.set_title("per label: sharp vs soft kernel")
    plt.tight_layout(); plt.show()

## 4. Comparing configurations

Rows with the same config hash are the same experiment with different seeds. Compare configurations on
**val** (that is what choices are made on); test is for reporting the chosen ones.

In [ ]:
group = ledger.groupby(["name", "config_hash", "aggregator", "loss", "lr"], dropna=False)
table = group[["val_macro_auroc", "val_macro_auprc", "best_epoch"]].agg(["mean", "std"])
table.columns = [f"{a}_{b}" for a, b in table.columns]
table.insert(0, "seeds", group.size())
display(table.sort_values("val_macro_auroc_mean", ascending=False).round(4))

fig, ax = plt.subplots(figsize=(10, 0.5 * len(table) + 1.5))
labels = [f"{n} [{h[:8]}]" for n, h, *_ in table.index]
for y, (key, g) in enumerate(group):
    ax.scatter(g.val_macro_auroc, [y] * len(g), color="#4C78A8")
ax.set_yticks(range(len(labels)), labels); ax.set_xlabel("val macro AUROC (one dot per seed)")
plt.tight_layout(); plt.show()